In [0]:
%pip install pandas
%pip install yfinance

In [0]:
import pandas as pd
import yfinance as yf

In [0]:
VOLUME = "/Volumes/raw/equity_dashboard_yfinance/files"
ingested_at = pd.Timestamp.now(tz="UTC")

tickers = ['META', 'AMZN', 'GOOGL']

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS raw")
spark.sql("CREATE SCHEMA IF NOT EXISTS raw.equity_dashboard_yfinance")
spark.sql("CREATE VOLUME IF NOT EXISTS raw.equity_dashboard_yfinance.files")

DataFrame[]

## Raw Dim Security

| Data type | Name |
| --- | --- |
| STRING | ticker |
| STRING | company_name |
| STRING | exchange |
| STRING | type |
| STRING | sector |
| STRING | industry |
| STRING | market |
| STRING | currency |
| TIMESTAMP | ingested_at |

In [0]:
raw_dim_security = []

for ticker in tickers:
  data = yf.Ticker(ticker)

  df = pd.DataFrame(
      {
          'ticker': data.ticker,
          'company_name': data.info.get('shortName'),
          'exchange': data.info.get('fullExchangeName'),
          'type': data.info.get('typeDisp'),
          'sector': data.info.get('sector'),
          'industry': data.info.get('industry'),
          'market': data.info.get('market'),
          'currency': data.info.get('currency'),
      },
      index=[0]
  )

  raw_dim_security.append(df)

raw_dim_security = pd.concat(raw_dim_security, ignore_index=True)
raw_dim_security

,ticker,company_name,exchange,type,sector,industry,market,currency
0,META,"Meta Platforms, Inc.",NasdaqGS,Equity,Communication Services,Internet Content & Information,us_market,USD
1,AMZN,"Amazon.com, Inc.",NasdaqGS,Equity,Consumer Cyclical,Internet Retail,us_market,USD
2,GOOGL,Alphabet Inc.,NasdaqGS,Equity,Communication Services,Internet Content & Information,us_market,USD


In [0]:
raw_dim_security["ingested_at"] = ingested_at
raw_dim_security.to_parquet(f'{VOLUME}/raw_dim_security.parquet', coerce_timestamps='us')
spark.sql(f"""
    CREATE OR REPLACE TABLE raw.equity_dashboard_yfinance.raw_dim_security AS
    SELECT * FROM read_files('{VOLUME}/raw_dim_security.parquet', format => 'parquet')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Raw Dim Date

| Data type | Name |
| --- | --- |
| DATE | date |
| INT | year |
| INT | quarter |
| INT | month |
| DATE | week_start |

In [0]:
end = pd.Timestamp.now().normalize()
dates = pd.date_range(start=end - pd.DateOffset(years=11), end=end, freq='D')

raw_dim_date = pd.DataFrame({'date': dates})
raw_dim_date['year'] = raw_dim_date['date'].dt.year
raw_dim_date['quarter'] = raw_dim_date['date'].dt.quarter
raw_dim_date['month'] = raw_dim_date['date'].dt.month
raw_dim_date['week_start'] = raw_dim_date['date'] - pd.to_timedelta(raw_dim_date['date'].dt.weekday, unit='D')

raw_dim_date

,date,year,quarter,month,week_start
0,2015-11-07,2015,4,11,2015-11-02
1,2015-11-08,2015,4,11,2015-11-02
2,2015-11-09,2015,4,11,2015-11-09
3,2015-11-10,2015,4,11,2015-11-09
4,2015-11-11,2015,4,11,2015-11-09
...,...,...,...,...,...
4014,2026-11-03,2026,4,11,2026-11-02
4015,2026-11-04,2026,4,11,2026-11-02
4016,2026-11-05,2026,4,11,2026-11-02
4017,2026-11-06,2026,4,11,2026-11-02


In [0]:
raw_dim_date['date'] = raw_dim_date['date'].dt.date
raw_dim_date['week_start'] = raw_dim_date['week_start'].dt.date
raw_dim_date.to_parquet(f'{VOLUME}/raw_dim_date.parquet', coerce_timestamps='us')
spark.sql(f"""
    CREATE OR REPLACE TABLE raw.equity_dashboard_yfinance.raw_dim_date AS
    SELECT * FROM read_files('{VOLUME}/raw_dim_date.parquet', format => 'parquet')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Raw Fact Price Daily

| Data type | Name |
| --- | --- |
| STRING | ticker |
| DATE | date |
| DOUBLE | open |
| DOUBLE | high |
| DOUBLE | low |
| DOUBLE | close |
| DOUBLE | adj_close |
| BIGINT | volume |
| DOUBLE | dividends |
| DOUBLE | stock_splits |
| TIMESTAMP | ingested_at |

In [0]:
raw_fact_price_daily = []

for ticker in tickers:
  data = yf.Ticker(ticker)
  price = data.history(period='11y', auto_adjust=False)
  price['ticker'] = data.ticker
  price = price.reset_index()

  # move ticker column to the front
  col = price.pop(price.columns[-1])
  price.insert(0, col.name, col)

  raw_fact_price_daily.append(price)

raw_fact_price_daily = pd.concat(raw_fact_price_daily, ignore_index=True)
raw_fact_price_daily = raw_fact_price_daily.rename(columns={
      'Date': 'date',
      'Open': 'open',
      'High': 'high',
      'Low': 'low',
      'Close': 'close',
      'Adj Close': 'adj_close',
      'Volume': 'volume',
      'Dividends': 'dividends',
      'Stock Splits': 'stock_splits',
    })

raw_fact_price_daily['date'] = raw_fact_price_daily['date'].dt.date
raw_fact_price_daily

,ticker,date,open,high,low,close,adj_close,volume,dividends,stock_splits
0,META,2015-11-09,107.000000,108.250000,105.860001,106.489998,105.479744,26383500,0.0,0.0
1,META,2015-11-10,105.959999,108.150002,105.239998,107.910004,106.886269,25031700,0.0,0.0
2,META,2015-11-11,108.269997,109.870003,107.110001,109.010002,107.975853,25057100,0.0,0.0
3,META,2015-11-12,108.480003,109.389999,108.000000,108.019997,106.995224,20924700,0.0,0.0
4,META,2015-11-13,107.709999,108.250000,103.830002,103.949997,102.963829,30278400,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
8290,GOOGL,2026-11-02,282.459991,287.000000,278.799988,278.989990,278.989990,48512300,0.0,0.0
8291,GOOGL,2026-11-03,280.230011,281.609985,277.200012,280.029999,280.029999,34974500,0.0,0.0
8292,GOOGL,2026-11-04,283.510010,296.489990,283.399994,288.929993,288.929993,51550300,0.0,0.0
8293,GOOGL,2026-11-05,293.970001,295.600006,277.070007,282.570007,282.570007,53412800,0.0,0.0


In [0]:
raw_fact_price_daily['ingested_at'] = ingested_at
raw_fact_price_daily.to_parquet(f'{VOLUME}/raw_fact_price_daily.parquet', coerce_timestamps='us')

spark.sql(f"""
    CREATE OR REPLACE TABLE raw.equity_dashboard_yfinance.raw_fact_price_daily AS
    SELECT * FROM read_files('{VOLUME}/raw_fact_price_daily.parquet', format => 'parquet')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Raw Fact Security Snapshot

| Data type | Name |
| --- | --- |
| STRING | ticker |
| DATE | snapshot_date |
| BIGINT | market_cap |
| DOUBLE | dividend_yield |
| BIGINT | volume |
| BIGINT | avg_volume |
| DOUBLE | beta |
| DOUBLE | pe |
| DOUBLE | forward_pe |
| DOUBLE | eps |
| DOUBLE | forward_eps |
| DOUBLE | peg_ratio |
| BIGINT | shares_outstanding |
| TIMESTAMP | ingested_at |


In [0]:
raw_fact_security_snapshot = []

for ticker in tickers:
  data = yf.Ticker(ticker)

  df = pd.DataFrame(
      {
          'ticker': data.ticker,
          'snapshot_date': pd.Timestamp.now().normalize(),
          'market_cap': data.info.get('marketCap'),
          'dividend_yield': data.info.get('dividendYield'),
          'volume': data.info.get('volume'),
          'avg_volume': data.info.get('averageVolume'),
          'beta': data.info.get('beta'),
          'pe': data.info.get('trailingPE'),
          'forward_pe': data.info.get('forwardPE'),
          'eps': data.info.get('trailingEps'),
          'forward_eps': data.info.get('forwardEps'),
          'peg_ratio': data.info.get('pegRatio'),
          'shares_outstanding': data.info.get('sharesOutstanding')
      },
      index=[0]
  )

  raw_fact_security_snapshot.append(df)

raw_fact_security_snapshot = pd.concat(raw_fact_security_snapshot, ignore_index=True)
raw_fact_security_snapshot

,ticker,snapshot_date,market_cap,dividend_yield,volume,avg_volume,beta,pe,forward_pe,eps,forward_eps,peg_ratio,shares_outstanding
0,META,2026-11-07,1560051712000,0.34,15657463,21030731,1.279,20.817130,17.144897,29.31,35.58785,0.79,2214455318
1,AMZN,2026-11-07,2803846610944,0.00,41256664,38646983,1.503,20.962004,23.781048,12.37,10.90364,1.21,10813138306
2,GOOGL,2026-11-07,3501085163520,0.31,36226983,28828813,1.226,14.871018,18.450144,19.15,15.43511,1.00,5888000000


In [0]:
raw_fact_security_snapshot['snapshot_date'] = raw_fact_security_snapshot['snapshot_date'].dt.date
raw_fact_security_snapshot["ingested_at"] = ingested_at
raw_fact_security_snapshot.to_parquet(f'{VOLUME}/raw_fact_security_snapshot.parquet', coerce_timestamps='us')

spark.sql(f"""
    CREATE OR REPLACE TABLE raw.equity_dashboard_yfinance.raw_fact_security_snapshot AS
    SELECT * FROM read_files('{VOLUME}/raw_fact_security_snapshot.parquet', format => 'parquet')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Raw Fact Price Target

| Data type | Name |
| --- | --- |
| STRING | ticker |
| DATE | snapshot_date |
| DOUBLE | current_price |
| DOUBLE | target_high |
| DOUBLE | target_low |
| DOUBLE | target_mean |
| DOUBLE | target_median |
| DOUBLE | recommendation_mean |
| STRING | recommendation_key |
| BIGINT | analyst_count |
| TIMESTAMP | ingested_at |

In [0]:
raw_fact_price_targets = []

for ticker in tickers:
  data = yf.Ticker(ticker)

  df = pd.DataFrame(
      {
          'ticker': data.ticker,
          'snapshot_date': pd.Timestamp.now().normalize(),
          'current_price': data.info.get('currentPrice'),
          'target_high': data.info.get('targetHighPrice'),
          'target_low': data.info.get('targetLowPrice'),
          'target_mean': data.info.get('targetMeanPrice'),
          'target_median': data.info.get('targetMedianPrice'),
          'recommendation_mean': data.info.get('recommendationMean'),
          'recommendation_key': data.info.get('recommendationKey'),
          'analyst_count': data.info.get('numberOfAnalystOpinions'),
      },
      index=[0]
  )

  raw_fact_price_targets.append(df)

raw_fact_price_targets = pd.concat(raw_fact_price_targets, ignore_index=True)
raw_fact_price_targets

,ticker,snapshot_date,current_price,target_high,target_low,target_mean,target_median,recommendation_mean,recommendation_key,analyst_count
0,META,2026-11-07,610.15,1050.0,629.06,848.37650,850.0,1.40323,strong_buy,58
1,AMZN,2026-11-07,259.30,405.0,230.00,326.85208,328.0,1.33333,strong_buy,58
2,GOOGL,2026-11-07,284.78,500.0,300.00,400.37357,400.0,1.34483,strong_buy,53


In [0]:
raw_fact_price_targets["snapshot_date"] = raw_fact_price_targets["snapshot_date"].dt.date
raw_fact_price_targets["ingested_at"] = ingested_at
raw_fact_price_targets.to_parquet(f'{VOLUME}/raw_fact_price_targets.parquet', coerce_timestamps='us')

spark.sql(f"""
    CREATE OR REPLACE TABLE raw.equity_dashboard_yfinance.raw_fact_price_targets AS
    SELECT * FROM read_files('{VOLUME}/raw_fact_price_targets.parquet', format => 'parquet')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]